### Project Introduction

Welcome to the Customer Analytics project!

#### About the Project

You and your team will work with real financial data from a small bank in Switzerland. The data is aggregated from 3 years of credit card transactions (2021–2023).

There are two different aggregations:

#### Customer Statistics

Statistics for each customer

| Column            | Description     |
|-------------------|-----------------|
|customer_id        | Unique Id                             |
|n_country          | Card used in number of countries      |
|n_transactions     | Number of transactions                |
|total_amount       | Total amount spent                    | 
|n_counterparts     | Number of unique counterparts         |
|days_active        | Number of days the card was active (last transaction - first transaction)     |
|pos_perc           | Point of sale (in store) percentage of total amount |     
|ecom_perc          | E-commerce (online) percentage of total amount |            
|                   |       |
| *Categories*      | Total amount spent per category                             |
|                   |       |
|education          |       |
|entertainment      |       |
|fitness            |       |
|...                |       |
|                   |       |
| *Currencies*          | Total amount spent in currency (only top currencies)       |
|                   |       |
|chf                |       |
|eur                |       |
| ...               |       |
|                   |       |
| *Countries*         | Total amount spent in country (only top countries)       |
|                   |       |
|ch                 |       |
|de                 |       |
| ...               |       |




#### Share of Wallet

Share of wallet for each month for each category or category and top counterpart for all customers.

| Column            | Description     |
|-------------------|-----------------|
|month              | Month of the year          |
|year               | Year           |
|category           | Category          |
|top_counterpart    | Top counterpart           |
|n_customers        | Number of individual customers who had a transaction per month/category/top_counterpart          |
|n_transasctions    | Number of individual transactions per month/category/top_counterpart          |
|total_amount       | Total amount spend per month/category/top_counterpart          |
|pos_perc           | Point of sale (in store) percentage of total amount |     
|ecom_perc          | E-commerce (online) percentage of total amount |      



# Exploratory Data Analysis - Customer Data Analytics Project

This notebook picks up after `example_code/example_code.ipynb` and looks at the three datasets in more depth before we build the churn model, the segmentation and the share-of-wallet story:

- `customer_data.csv` - 5,576 customers, 42 aggregated features (2021-2023 credit card transactions)
- `customer_data_labels.csv` / `customer_data_predict.csv` - the 70/30 churn split (3,903 labeled, 1,673 to predict)
- `sow_category.csv` / `sow_category_counterpart.csv` - monthly share-of-wallet by category (and category + top counterpart)

Goal of this pass: understand data quality, distributions, and the relationship between customer behavior and churn, so the modeling and segmentation steps that follow are built on solid ground.

In [ ]:
import subprocess
import sys
from pathlib import Path

# repo root, whether the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROCESSED = ROOT / "data" / "raw", ROOT / "data" / "processed"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Make sure the cleaned data exists before trying to load it. If this is a
# fresh clone (or data/raw/ was just updated), run src/clean_data.py once;
# on later runs data/processed/ is already there, so this is skipped.
processed_files = [PROCESSED / "customer_data_clean.csv", PROCESSED / "sow_category_clean.csv"]
if not all(f.exists() for f in processed_files):
    print("data/processed/ missing or incomplete, running src/clean_data.py ...")
    subprocess.run([sys.executable, str(ROOT / "src" / "clean_data.py")], check=True, cwd=ROOT)
else:
    print("data/processed/ already up to date, skipping src/clean_data.py.")

# Load the cleaned data produced by src/clean_data.py (data/processed/).
# labels/predict and sow_category_counterpart have no cleaned version yet,
# so those are still read straight from data/raw/.
cust = pd.read_csv(PROCESSED / "customer_data_clean.csv")
labels = pd.read_csv(RAW / "customer_data_labels.csv")
predict = pd.read_csv(RAW / "customer_data_predict.csv")
sow_cat = pd.read_csv(PROCESSED / "sow_category_clean.csv")
sow_cp = pd.read_csv(RAW / "sow_category_counterpart.csv")

labeled = cust.merge(labels, on="customer_id")
unlabeled = cust.merge(predict, on="customer_id")

print(cust.shape, labels.shape, predict.shape, sow_cat.shape, sow_cp.shape)

## 1. Data quality checks

This notebook reads the **cleaned** data produced by `src/clean_data.py` (`data/processed/customer_data_clean.csv` and `data/processed/sow_category_clean.csv`), so the fixes below are already applied here, not redone in this notebook. They're listed for reference; see the script for the exact logic and rationale.

- **No missing values** in any of the five files, and **no duplicate `customer_id`s** in the customer table.
- `customer_data_labels.csv` (3,903 rows) and `customer_data_predict.csv` (1,673 rows) partition `customer_data.csv` exactly (3,903 + 1,673 = 5,576, no overlap), matching the 70/30 split from the slides.
- `pos_perc + ecom_perc` sums to 1.0 for every customer (max 1.01, a rounding artifact), so the two are complementary as expected.
- **Category naming was inconsistent in the raw SOW file.** `sow_category.csv` originally mixed lowercase and capitalized versions of the same category for a handful of rows (`Groceries`/`groceries`, `Restaurant`/`restaurants`, `Fitness`/`fitness`, `Health`/`health`, `Holidays`/`holidays`, `Hotel`, `Shopping`, `Lebensmittel`). `src/clean_data.py` lower-cases and folds these together before re-aggregating, so `sow_category_clean.csv` has 18 categories instead of the raw file's 24, and `pos_perc`/`ecom_perc` are recomputed as amount-weighted averages rather than summed.
- **`cat_lebensmittel` and `cat_restaurant` in the raw customer file were mislabeled duplicates**, not real categories: `cat_lebensmittel` (German for "groceries") totaled only CHF 1,097 across a single customer, and `cat_restaurant` (singular) totaled CHF 4,583 across 67 customers, both negligible next to `cat_groceries` and `cat_restaurants`. `src/clean_data.py` folds them into the real category and drops the columns, so `customer_data_clean.csv` has 40 columns instead of the raw file's 42.
- **The raw SOW time series had a noisy first month.** `sow_category.csv` included a `12-2020` entry (3 rows, ~CHF 378 total), one month before the stated 2021-2023 window. `src/clean_data.py` drops it.
- `n_country` ("card used in number of countries") ranges from 1 to 33 with a long right tail, almost certainly reflecting merchant billing country (e.g. online purchases from foreign e-commerce sites) rather than physical travel, since a median customer is in the 1-3 country range. This one is a real signal, not a data error, so it wasn't touched by the cleaning script.

In [ ]:
num_cols = ["n_country", "n_transactions", "total_amount", "n_counterparts", "days_active", "pos_perc", "ecom_perc"]
cust[num_cols].describe().round(2)

**Distributions are heavily right-skewed** (typical for transaction data): median `total_amount` is CHF 1,865 but the mean is CHF 10,799, pulled up by a long tail of high-volume customers (max CHF 333k, 5,732 transactions for the single most active customer). `days_active` spans the full 1-964 day range of the observation window, which matters a lot for churn (see below) - a customer who joined recently will have low `days_active` and few transactions purely because they haven't had time to accumulate them, not because they're disengaged.

In [ ]:
category_cols = [c for c in cust.columns if c.startswith("cat_")]
clean_cat = cust[category_cols].sum().sort_values(ascending=False)
clean_cat.index = clean_cat.index.str.replace("cat_", "", regex=False)

clean_cat.head(10).plot(kind="bar", figsize=(10,5), title="Total spend by category (cleaned)")
plt.show()

**Shopping, groceries and restaurants dominate spend** (CHF 11.1M, 10.4M and 8.0M respectively after merging the mislabeled duplicates), followed by cash withdrawals and holidays. This roughly matches the "Retail SOW" framing from the brief - groceries alone is worth a dedicated share-of-wallet story (Coop/Migros/Aldi/Lidl, as shown in the example slides).

In [ ]:
total_spending = cust["total_amount"].sum()
grocery_spending = cust["cat_groceries"].sum()
share_of_wallet_groceries = grocery_spending / total_spending * 100

print(f"Total spending: CHF {total_spending:,.2f}")
print(f"Grocery spending: CHF {grocery_spending:,.2f}")
print(f"Share of wallet for groceries: {share_of_wallet_groceries:.1f}%")

This is the same "share of wallet" idea as the individual-customer example in the project brief (e.g. "Coop has a SOW of 30.3%"), just computed across the whole customer base instead of one person: of every CHF spent by these 5,576 customers over the three years, 17.3% went to groceries. It lines up with the monthly SOW chart further down, where groceries sits in a fairly narrow 15-20% band throughout, so the aggregate number and the time series agree with each other.

## 2. Churn overview

`churned` is available for 3,903 of the 5,576 customers (the other 1,673 are what we need to predict). The label is fairly balanced:

In [ ]:
print(labeled["churned"].value_counts())
print("Churn rate:", labeled["churned"].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4.5))
for ax, col in zip(axes, ["n_transactions", "days_active", "total_amount"]):
    sns.boxplot(data=labeled, x="churned", y=col, hue="churned", ax=ax, showfliers=False, legend=False)
    if col != "days_active":
        ax.set_yscale("log")
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Churned customers look like much less engaged customers across the board**: on average they have far fewer transactions (78 vs. 418), spend far less (CHF 3.7k vs. CHF 19.1k), have fewer counterparts (28 vs. 134), and a much shorter active window (229 vs. 594 days). `days_active` is the single strongest correlate of churn (r = -0.54), followed by `n_country` (-0.47) and `n_counterparts` (-0.43). None of the individual spending categories correlate meaningfully with churn on their own (all |r| < 0.08) - churn looks driven by overall activity level and breadth (how many merchants/countries/transactions), not by what a customer buys.

**Important caveat for modeling**: `days_active` is defined as `last_transaction - first_transaction`. A customer who joined the bank recently will have a short `days_active` window *by construction*, independent of whether they're actually "churning" in the behavioral sense. Since this is also the strongest predictor, it's worth checking (or asking) whether `churned` was labeled using a similar recency-based rule - if so, a model built on these features may just be re-learning the label definition rather than finding an actionable early-warning signal. Worth flagging in the write-up regardless of how the model turns out.

## 3. Share of wallet over time

Using the cleaned `sow_category_clean.csv` (categories already lower-cased and merged, noisy December 2020 stub already dropped by `src/clean_data.py`), the six largest categories account for the bulk of aggregate spend. Plotting their monthly share of total spend (January 2021 to December 2023):

In [ ]:
def parse_ym(s):
    m, y = s.split("-")
    return int(y) * 12 + int(m)

sow_cat["ym_sort"] = sow_cat["year_month"].apply(parse_ym)
sow_cat = sow_cat.sort_values("ym_sort")

monthly_total = sow_cat.groupby("ym_sort")["total_amount"].sum()
top6 = sow_cat.groupby("category")["total_amount"].sum().sort_values(ascending=False).head(6).index.tolist()

pivot = sow_cat[sow_cat["category"].isin(top6)].pivot_table(
    index="ym_sort", columns="category", values="total_amount", aggfunc="sum", fill_value=0)
share = pivot.div(monthly_total.reindex(pivot.index), axis=0) * 100
share.index = sow_cat.drop_duplicates("ym_sort").set_index("ym_sort").loc[share.index, "year_month"]

share.plot(figsize=(13,5.5), marker="o", markersize=3, title="Share of wallet by category over time")
plt.ylabel("Share of wallet (%)")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 4. Merchant-level share of wallet

`sow_category_counterpart.csv` breaks the same spend down one level further: category *and* top counterpart (merchant), grouped by month. Per the project brief, only counterparts with at least 1,000 transactions are broken out individually, everything else is grouped under `"other"`. A quick look at the file before charting it:

In [ ]:
sow_cp = pd.read_csv(RAW / "sow_category_counterpart.csv")
print(sow_cp.shape)
print("Distinct named counterparts:", sow_cp["top_counterpart"].nunique())
print("Rows grouped under 'other':", (sow_cp["top_counterpart"] == "other").sum())
sow_cp.head()

In [ ]:
top_counterparts = (
    sow_cp[sow_cp["top_counterpart"] != "other"]
    .groupby("top_counterpart")["total_amount"].sum()
    .sort_values(ascending=False)
    .head(20)
)
top_counterparts.plot(kind="bar", figsize=(13, 5.5), title="Top 20 merchants by total spend (2021-2023)")
plt.ylabel("Total spend (CHF)")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

**Coop (CHF 3.64M) and Migros (CHF 2.93M) are the two biggest counterparts by far**, consistent with them dominating the groceries category in the brief's own single-customer example. What's more interesting is who shows up right behind them: Revolut (CHF 2.47M), PayPal (CHF 1.15M), Binance (CHF 0.66M) and PostFinance (CHF 0.60M) are all in the top 10, and none of them are retailers, they're payment platforms, neobanks and a crypto exchange. That's a real signal about *this* bank's customers specifically (multi-banking and crypto activity are common enough to show up at an aggregate level), and also a modeling caveat: transactions routed through Revolut or PayPal are the underlying purchase re-categorized by an intermediary, so "top counterpart" isn't always the same thing as "where the money actually ended up." Worth keeping in mind if this file is used for a retailer-facing share-of-wallet story rather than a bank-facing one.

A few patterns stand out that could seed the "History and Future" / "Different Categories" angles from the brief:

- **Shopping's share has been gradually declining** relative to other categories (from ~25% in early 2021 toward ~17-19% by 2023) even though absolute shopping spend keeps growing - other categories are growing faster.
- **Restaurants and holidays show clear seasonality**, both dipping in winter months and peaking around summer (restaurants) or mid/late summer (holidays) - consistent with post-COVID travel and dining recovery through 2021-2023.
- **Cash's share has crept up over the period** (roughly 3-5% in 2021 to 13-18% by 2023), which is worth a closer look - it could reflect genuine behavior change or just a shift in how certain transactions get categorized.
- **Groceries is the most stable category**, sitting in a fairly narrow 15-20% band throughout - a defensive, non-discretionary share of wallet.

